# Leakage Diagnostics

Six targeted checks run after the initial modeling pass to validate that headline scores reflect content discrimination rather than artifacts:

1. **Exact duplicates** between train and test
2. **Near-duplicates** via embedding cosine similarity
3. **Cross-source topical proximity** — e.g. MMLU-bio test rows that are close to WMDP train rows
4. **Label sanity** — manual spot-check of test labels per bucket
5. **Lexical signal strength** — top discriminating features per class (distinguishes "task is genuinely separable" from "model is detecting dataset provenance")
6. **Bucket-pair separability** — how separable is each negative bucket from positives, individually

CPU is sufficient; ~3–5 minutes end-to-end.

## Environment setup

In [ ]:
!pip install -q sentence-transformers scikit-learn pandas pyarrow

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
DATA_DIR = Path("/content/drive/MyDrive/biology_refusal/data")
OUT_DIR  = Path("/content/drive/MyDrive/biology_refusal/models")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

train_df = pd.read_parquet(DATA_DIR / "train.parquet")
test_df  = pd.read_parquet(DATA_DIR / "test.parquet")
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

## Check 1 — Exact duplicates between train and test

In [ ]:
# If any test text exactly matches any train text, that's a hard leakage bug.
# We did dedup at the end of the pipeline but worth verifying directly.

print("\n" + "=" * 70)
print("CHECK 1: Exact train/test text duplicates")
print("=" * 70)

train_texts = set(train_df["text"])
test_texts  = set(test_df["text"])
overlap = train_texts & test_texts
print(f"  Train unique texts: {len(train_texts)}")
print(f"  Test unique texts:  {len(test_texts)}")
print(f"  Exact overlap:      {len(overlap)}")

if overlap:
    print("\n  LEAKED EXAMPLES (first 5):")
    for t in list(overlap)[:5]:
        print(f"    - {t[:150]}")
    # Where do they come from?
    leaked_train = train_df[train_df["text"].isin(overlap)]
    leaked_test  = test_df[test_df["text"].isin(overlap)]
    print(f"\n  Sources of leaked texts in train:\n{leaked_train['source'].value_counts()}")
    print(f"\n  Sources of leaked texts in test:\n{leaked_test['source'].value_counts()}")
else:
    print("  ✓ No exact duplicates between train and test.")

## Check 2 — Near-duplicates via embedding similarity

In [ ]:
# Cosine similarity between every test example and every train example.
# Anything above 0.95 is suspicious; above 0.98 is almost certainly a near-dup.
#
# We do this with the SAME encoder we used for the modeling — anything that
# embeds close enough to a train example for the encoder to recognize is
# effectively leaked for downstream models built on those embeddings.

print("\n" + "=" * 70)
print("CHECK 2: Near-duplicates via embedding cosine similarity")
print("=" * 70)

encoder = SentenceTransformer("all-MiniLM-L6-v2")

print("  Encoding train...")
train_emb = encoder.encode(
    train_df["text"].tolist(), batch_size=64,
    show_progress_bar=False, convert_to_numpy=True,
)
print("  Encoding test...")
test_emb = encoder.encode(
    test_df["text"].tolist(), batch_size=64,
    show_progress_bar=False, convert_to_numpy=True,
)

# Compute pairwise similarity, find max train similarity for each test example.
print("  Computing pairwise similarities...")
sim_matrix = cosine_similarity(test_emb, train_emb)
max_sim_per_test = sim_matrix.max(axis=1)
nearest_train_idx = sim_matrix.argmax(axis=1)

# Report distribution.
print(f"\n  Distribution of max train-similarity per test example:")
for pct in [50, 75, 90, 95, 99]:
    val = np.percentile(max_sim_per_test, pct)
    print(f"    p{pct}: {val:.4f}")
print(f"    max:  {max_sim_per_test.max():.4f}")

# Flag high-similarity pairs.
THRESHOLDS = [0.99, 0.95, 0.90]
for thr in THRESHOLDS:
    n_above = (max_sim_per_test >= thr).sum()
    print(f"  Test examples with max train-sim >= {thr}: {n_above} ({100*n_above/len(test_df):.1f}%)")

# Show the top 10 highest-similarity test/train pairs.
top_indices = np.argsort(-max_sim_per_test)[:10]
print("\n  TOP 10 highest train-similarity test examples:")
for rank, test_idx in enumerate(top_indices, start=1):
    train_idx = nearest_train_idx[test_idx]
    sim = max_sim_per_test[test_idx]
    test_row  = test_df.iloc[test_idx]
    train_row = train_df.iloc[train_idx]
    print(f"\n  [{rank}] sim={sim:.4f}")
    print(f"      TEST  ({test_row['source']}, label={test_row['label']}):  {test_row['text'][:160]}")
    print(f"      TRAIN ({train_row['source']}, label={train_row['label']}): {train_row['text'][:160]}")

## Check 3 — Cross-bucket topical leakage on WMDP-bio specifically

In [ ]:
# Our cluster-based holdout protected WMDP-positive train from WMDP-positive
# test topically. But the model could be exploiting topical proximity in
# either direction:
#   (a) MMLU-bio test questions that are very close to WMDP train questions
#       — model might've over-fit and refuse them.
#   (b) WMDP test questions that are very close to MMLU-bio train questions
#       — easier to misclassify as benign.
# Look at the cross-source similarity distribution specifically.

print("\n" + "=" * 70)
print("CHECK 3: Cross-source topical proximity")
print("=" * 70)

def cross_source_max_sim(test_source, train_source):
    """Max embedding similarity between test rows of `test_source` and
    train rows of `train_source`."""
    test_mask  = test_df["source"]  == test_source
    train_mask = train_df["source"] == train_source
    if test_mask.sum() == 0 or train_mask.sum() == 0:
        return None
    sub_sim = cosine_similarity(test_emb[test_mask], train_emb[train_mask])
    return sub_sim.max(axis=1)

source_pairs = [
    ("wmdp_bio_original", "wmdp_bio_original"),  # baseline: positive train to positive test
    ("wmdp_bio_original", "mmlu_bio"),           # WMDP test -> bio train negatives
    ("mmlu_bio", "wmdp_bio_original"),           # MMLU-bio test -> WMDP train positives
    ("mmlu_bio", "mmlu_bio"),                    # within MMLU-bio
    ("dual_use", "wmdp_bio_original"),           # dual_use test -> WMDP train positives
    ("dual_use", "dual_use"),                    # within dual_use
]

print(f"\n  {'test_source':<22} {'train_source':<22} {'median':>8} {'p90':>8} {'p99':>8} {'max':>8}")
print(f"  {'-'*22} {'-'*22} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
for test_src, train_src in source_pairs:
    sims = cross_source_max_sim(test_src, train_src)
    if sims is None:
        continue
    print(f"  {test_src:<22} {train_src:<22} "
          f"{np.median(sims):>8.4f} {np.percentile(sims, 90):>8.4f} "
          f"{np.percentile(sims, 99):>8.4f} {sims.max():>8.4f}")

# Interpretation guide:
# - Within-source similarity (e.g. WMDP test -> WMDP train) should be HIGHER
#   than cross-source. If cross-source max is comparable to within-source max,
#   we have topical contamination.
# - The (mmlu_bio test, wmdp train) row is the critical one. If MMLU-bio test
#   questions are very close to WMDP train questions, the model might be
#   confusing topical proximity for the refusal label.

## Check 4 — Label sanity (manual spot check)

In [ ]:
# Print 5 random examples from each bucket. Read them and confirm the labels
# are sensible. Catches obvious labeling bugs.

print("\n" + "=" * 70)
print("CHECK 4: Label sanity spot-check (5 random per bucket)")
print("=" * 70)

for bucket in sorted(test_df["source"].unique()):
    sub = test_df[test_df["source"] == bucket].sample(
        n=min(5, (test_df["source"] == bucket).sum()), random_state=42
    )
    label = sub["label"].iloc[0]
    print(f"\n  --- {bucket} (label={label}) ---")
    for _, row in sub.iterrows():
        print(f"    • {row['text'][:200]}")

## Check 5 — Lexical signal strength

In [ ]:
# Train a TF-IDF + LR on JUST the train set, then look at the top features for
# each class. If the top "refuse" features are obvious giveaway words that
# only appear on one side of the label (e.g. specific technical terms that
# never appear in negatives), that explains the high accuracy without it
# being a "leakage bug" per se — just that the task is lexically separable.

print("\n" + "=" * 70)
print("CHECK 5: Top discriminating lexical features (TF-IDF + LR)")
print("=" * 70)

from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(
    ngram_range=(1, 1), min_df=3, max_df=0.95, max_features=20000,
)
X_tr = vec.fit_transform(train_df["text"])
y_tr = train_df["label"].to_numpy()

lr = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
lr.fit(X_tr, y_tr)

feature_names = np.array(vec.get_feature_names_out())
coefs = lr.coef_[0]

print("\n  Top 25 features pushing toward REFUSE (label=1):")
top_pos = np.argsort(-coefs)[:25]
for i in top_pos:
    print(f"    {feature_names[i]:30s}  coef={coefs[i]:+.3f}")

print("\n  Top 25 features pushing toward DON'T REFUSE (label=0):")
top_neg = np.argsort(coefs)[:25]
for i in top_neg:
    print(f"    {feature_names[i]:30s}  coef={coefs[i]:+.3f}")

# If top REFUSE features include source-name-like tokens, formatting artifacts,
# or specific named entities that only appear in WMDP, the task is genuinely
# lexically separable. That's not leakage but it does explain the numbers.

## Check 6 — Bucket-pair separability

In [ ]:
# How separable is each negative bucket from positives, on its own?
# If alpaca is trivially separable but dual_use isn't, and DistilBERT's
# aggregate F1 is high mostly because alpaca dominates the negative set,
# the headline numbers are misleading.

print("\n" + "=" * 70)
print("CHECK 6: Pairwise separability (positives vs each negative bucket)")
print("=" * 70)

from sklearn.linear_model import LogisticRegressionCV

pos_train = train_df[train_df["label"] == 1]
for neg_bucket in sorted(train_df[train_df["label"] == 0]["source"].unique()):
    neg_train = train_df[train_df["source"] == neg_bucket]
    pos_test = test_df[test_df["label"] == 1]
    neg_test = test_df[test_df["source"] == neg_bucket]

    # Train a tiny lexical classifier on JUST positives vs this one neg bucket.
    tr_texts  = pd.concat([pos_train["text"],  neg_train["text"]])
    tr_labels = np.concatenate([np.ones(len(pos_train)), np.zeros(len(neg_train))])
    te_texts  = pd.concat([pos_test["text"],   neg_test["text"]])
    te_labels = np.concatenate([np.ones(len(pos_test)),  np.zeros(len(neg_test))])

    vec_pair = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=20000)
    X_tr_pair = vec_pair.fit_transform(tr_texts)
    X_te_pair = vec_pair.transform(te_texts)
    clf = LogisticRegressionCV(Cs=5, cv=3, max_iter=1000, class_weight="balanced", n_jobs=-1)
    clf.fit(X_tr_pair, tr_labels)
    acc = clf.score(X_te_pair, te_labels)
    print(f"  positives vs {neg_bucket:20s}  test_acc = {acc:.4f}  "
          f"(n_pos_te={len(pos_test)}, n_neg_te={len(neg_test)})")

# Interpretation: if all four are >0.99 with a simple TF-IDF classifier,
# the task really is mostly lexically separable, and the modeling headline
# numbers reflect that fact rather than any leakage. If some buckets are
# 0.95 and others 0.99, our bucket difficulty stratification is working.